# Project 1 — Production-Grade RAG

Self-contained. **Just run top to bottom.** Needs `OPENAI_API_KEY` in `../.env`.
It writes its own sample corpus + golden eval set, runs all 3 phases, and emits **`RESULTS.md`** with the performance numbers.

1. Fundamentals: ingest → chunk → embed → retrieve → cited answer
2. Production: hybrid (BM25 + vector) + cross-encoder re-ranker + citation enforcement
3. Shippable: golden eval set + ragas faithfulness + CI gate

In [ ]:
%pip install -q langchain langchain-community langchain-openai chromadb rank_bm25 \
    sentence-transformers ragas datasets python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')
assert os.getenv('OPENAI_API_KEY'), 'Put OPENAI_API_KEY in ../.env (see ../.env.example)'

### Sample corpus (written automatically)
A small fictional knowledge base for **"Acme Cloud"** so retrieval has real, verifiable content. Replace `./corpus` with your own PDFs/markdown anytime — no code change needed.

In [ ]:
import os, textwrap
os.makedirs('./corpus', exist_ok=True)

CORPUS = {
  'pricing.md': '''# Acme Cloud Pricing
Acme Cloud has three tiers. The Free tier includes 1 project, 500 MB storage, and 10,000 API calls per month at no cost.
The Pro tier costs $29 per month and includes 10 projects, 50 GB storage, and 2 million API calls per month.
The Enterprise tier is custom-priced and includes unlimited projects, dedicated support, a 99.99% uptime SLA, and SSO.
Overage on the Pro tier is billed at $0.50 per additional GB of storage.''',
  'auth.md': '''# Acme Cloud Authentication
Acme Cloud supports API key authentication and OAuth 2.0. API keys are passed in the Authorization header as a Bearer token.
API keys can be rotated from the dashboard under Settings > API Keys. A rotated key remains valid for a 24-hour grace period.
OAuth 2.0 supports the authorization code flow. Access tokens expire after 1 hour; refresh tokens expire after 30 days.
Enterprise customers can enable SAML-based Single Sign-On (SSO) with Okta and Azure AD.''',
  'limits.md': '''# Acme Cloud Rate Limits
The default rate limit is 100 requests per second per API key. Exceeding it returns HTTP 429 with a Retry-After header.
Pro tier customers can request a limit increase up to 500 requests per second by contacting support.
Batch endpoints accept up to 1,000 items per request. Webhook delivery is retried up to 5 times with exponential backoff.''',
  'regions.md': '''# Acme Cloud Regions and Data Residency
Acme Cloud operates in four regions: us-east-1, us-west-2, eu-central-1, and ap-southeast-1.
Data residency is guaranteed within the chosen region. eu-central-1 is GDPR-compliant and located in Frankfurt, Germany.
Cross-region replication is available on the Enterprise tier only. The default region for new projects is us-east-1.'''
}
for name, body in CORPUS.items():
    with open(f'./corpus/{name}', 'w') as f:
        f.write(textwrap.dedent(body))
print('Wrote', len(CORPUS), 'docs to ./corpus')

## Phase 1 — Fundamentals
Chunk **~500-800 tokens, ~100 overlap** → embed → store → top-k → cited answer.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

docs = []
for f in os.listdir('./corpus'):
    docs += TextLoader(f'./corpus/{f}').load()

# ~500-800 tokens ≈ 2000-3200 chars; ~100 token overlap ≈ 400 chars
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
chunks = splitter.split_documents(docs)
print(f'{len(docs)} docs -> {len(chunks)} chunks')

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma.from_documents(chunks, embeddings, collection_name='acme',
                                    persist_directory='./chroma_db')
retriever = vectorstore.as_retriever(search_kwargs={'k': 5})
print('Vector store ready:', vectorstore._collection.count(), 'vectors')

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

# Prompt versioned as a named constant (treat prompts as architecture)
ANSWER_PROMPT = ChatPromptTemplate.from_template(
    'Answer ONLY from the context. Cite chunk numbers like [1], [2].\n'
    'If the context does not support an answer, reply exactly: I cannot answer this from the documents.\n\n'
    'Context:\n{context}\n\nQuestion: {question}'
)
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

def answer_v1(question):
    hits = retriever.invoke(question)
    ctx = '\n\n'.join(f'[{i+1}] {d.page_content}' for i, d in enumerate(hits))
    return llm.invoke(ANSWER_PROMPT.format(context=ctx, question=question)).content, hits

ans, sources = answer_v1('How much does the Pro tier cost and what does it include?')
print(ans)
print('--- sources ---')
for i, s in enumerate(sources):
    print(f'[{i+1}]', s.metadata.get('source'))

## Phase 2 — Production Quality
Hybrid retrieval (BM25 + vector) + cross-encoder re-ranker + citation enforcement.

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from sentence_transformers import CrossEncoder

bm25 = BM25Retriever.from_documents(chunks); bm25.k = 5
hybrid = EnsembleRetriever(retrievers=[bm25, retriever], weights=[0.4, 0.6])
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def retrieve_reranked(question, top_n=4):
    candidates = hybrid.invoke(question)
    if not candidates:
        return []
    scores = reranker.predict([[question, d.page_content] for d in candidates])
    ranked = [d for _, d in sorted(zip(scores, candidates), key=lambda x: -x[0])]
    return ranked[:top_n]

def answer_v2(question):
    hits = retrieve_reranked(question)
    if not hits:
        return 'I cannot answer this from the documents.', []
    ctx = '\n\n'.join(f'[{i+1}] {d.page_content}' for i, d in enumerate(hits))
    return llm.invoke(ANSWER_PROMPT.format(context=ctx, question=question)).content, hits

# Citation enforcement check: out-of-domain question should be declined
print(answer_v2('What is the capital of France?')[0])

## Phase 3 — Golden Eval + Faithfulness + CI Gate
20 manually-verified Q&A pairs (answers checked against the corpus above).

In [ ]:
GOLDEN = [
  {'question': 'How much does the Pro tier cost per month?', 'ground_truth': 'The Pro tier costs $29 per month.'},
  {'question': 'How many API calls does the Free tier include?', 'ground_truth': 'The Free tier includes 10,000 API calls per month.'},
  {'question': 'What storage does the Pro tier include?', 'ground_truth': 'The Pro tier includes 50 GB of storage.'},
  {'question': 'What is the storage overage charge on Pro?', 'ground_truth': 'Overage is billed at $0.50 per additional GB.'},
  {'question': 'What uptime SLA does Enterprise offer?', 'ground_truth': 'Enterprise offers a 99.99% uptime SLA.'},
  {'question': 'How are API keys passed?', 'ground_truth': 'As a Bearer token in the Authorization header.'},
  {'question': 'How long is the grace period for a rotated API key?', 'ground_truth': 'A rotated key remains valid for a 24-hour grace period.'},
  {'question': 'When do OAuth access tokens expire?', 'ground_truth': 'Access tokens expire after 1 hour.'},
  {'question': 'When do refresh tokens expire?', 'ground_truth': 'Refresh tokens expire after 30 days.'},
  {'question': 'Which identity providers does SSO support?', 'ground_truth': 'SSO supports Okta and Azure AD via SAML.'},
  {'question': 'What is the default rate limit?', 'ground_truth': '100 requests per second per API key.'},
  {'question': 'What HTTP status is returned when the rate limit is exceeded?', 'ground_truth': 'HTTP 429 with a Retry-After header.'},
  {'question': 'What rate limit can Pro customers request?', 'ground_truth': 'Up to 500 requests per second.'},
  {'question': 'How many items can a batch endpoint accept?', 'ground_truth': 'Up to 1,000 items per request.'},
  {'question': 'How many times are webhooks retried?', 'ground_truth': 'Up to 5 times with exponential backoff.'},
  {'question': 'Which regions does Acme Cloud operate in?', 'ground_truth': 'us-east-1, us-west-2, eu-central-1, and ap-southeast-1.'},
  {'question': 'Where is the GDPR-compliant region located?', 'ground_truth': 'eu-central-1, in Frankfurt, Germany.'},
  {'question': 'Which tier offers cross-region replication?', 'ground_truth': 'Only the Enterprise tier.'},
  {'question': 'What is the default region for new projects?', 'ground_truth': 'us-east-1.'},
  {'question': 'How many projects does the Free tier allow?', 'ground_truth': 'The Free tier includes 1 project.'},
]
print(len(GOLDEN), 'golden Q&A pairs')

In [ ]:
from datasets import Dataset
rows = []
for item in GOLDEN:
    ans, hits = answer_v2(item['question'])
    rows.append({'question': item['question'], 'answer': ans,
                 'contexts': [d.page_content for d in hits] or ['(none)'],
                 'ground_truth': item['ground_truth']})
ds = Dataset.from_list(rows)
print('Built eval dataset:', len(ds), 'rows')

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision

result = evaluate(ds, metrics=[faithfulness, answer_relevancy, context_precision])
scores = {k: float(v) for k, v in result.items() if isinstance(v, (int, float))}
print(scores)

In [ ]:
# CI gate + write RESULTS.md
THRESHOLD = 0.80
faith = scores.get('faithfulness', 0.0)
passed = faith >= THRESHOLD

md = ['# Project 1 — RAG Performance Report', '',
      f'Corpus: {len(docs)} docs / {len(chunks)} chunks. Eval set: {len(GOLDEN)} verified Q&A pairs.',
      '', '## Metrics (ragas)', '', '| Metric | Score |', '|---|---|']
for k, v in scores.items():
    md.append(f'| {k} | {v:.3f} |')
md += ['', f'## CI Gate', '',
       f'Threshold: faithfulness >= {THRESHOLD}', '',
       f'**Result: {"PASS ✅" if passed else "FAIL ❌"}** (faithfulness = {faith:.3f})', '',
       '## Pipeline', '- Phase 1: top-k vector retrieval + cited answers',
       '- Phase 2: hybrid BM25+vector + cross-encoder rerank + citation enforcement',
       '- Phase 3: golden eval + ragas faithfulness wired as a CI gate']
with open('RESULTS.md', 'w') as f:
    f.write('\n'.join(md))
print('Wrote RESULTS.md')
assert passed, f'CI GATE FAILED: faithfulness {faith:.3f} < {THRESHOLD}'
print('CI gate PASSED')